In [ ]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredLogarithmicError

In [ ]:
# 2. Load ECG Dataset
url = "http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv"
df = pd.read_csv(url)

features = df.iloc[:, :-1]
target = df.iloc[:, -1]

print("Dataset Shape:", df.shape)
print("Target Classes:\n", target.value_counts())

In [ ]:
# 3. Split Dataset
x_train, x_test, y_train, y_test = train_test_split(
    features, target,
    test_size=0.2,
    stratify=target,
    random_state=42
)

In [ ]:
# 4. Select Normal ECG Samples
x_train = x_train[y_train == 1]

print("Normal Training Shape:", x_train.shape)

In [ ]:
# 5. Normalize Data
scaler = MinMaxScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
# 6. Create AutoEncoder Model
class AutoEncoder(Model):

    def __init__(self, output_units):
        super().__init__()

        self.encoder = Sequential([
            Dense(128, activation='relu'),
            Dense(64, activation='relu'),
            Dense(32, activation='relu'),
            Dense(16, activation='relu')
        ])

        self.decoder = Sequential([
            Dense(32, activation='relu'),
            Dense(64, activation='relu'),
            Dense(128, activation='relu'),
            Dense(output_units, activation='sigmoid')
        ])

    def call(self, inputs):
        encoded = self.encoder(inputs)
        return self.decoder(encoded)

In [ ]:
# 7. Build and Compile Model
model = AutoEncoder(x_train_scaled.shape[1])
model.build(input_shape=(None, x_train_scaled.shape[1]))
model.summary()

model.compile(
    optimizer=Adam(),
    loss=MeanSquaredLogarithmicError(),
    metrics=['mse']
)

In [ ]:
# 8. Train Model
history = model.fit(
    x_train_scaled,
    x_train_scaled,
    epochs=20,
    batch_size=512,
    validation_data=(x_test_scaled, x_test_scaled)
)

In [ ]:
# 9. Plot Training Loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSLE")
plt.legend()
plt.show()

In [ ]:
# 10. Calculate Reconstruction Error
train_reconstructions = model.predict(x_train_scaled, verbose=0)

reconstruction_errors = tf.keras.losses.msle(
    x_train_scaled, train_reconstructions
)

# Calculate Anomaly Threshold
threshold = (
    np.mean(reconstruction_errors.numpy())
    + np.std(reconstruction_errors.numpy())
)

print("Anomaly Threshold:", threshold)

In [ ]:
# 11. Detect Anomalies
test_reconstructions = model.predict(x_test_scaled, verbose=0)

test_errors = tf.keras.losses.msle(
    x_test_scaled, test_reconstructions
)

predicted_labels = (
    test_errors.numpy() > threshold
).astype(int)

In [ ]:
# 12. Calculate Accuracy
# Dataset: 1 = Normal, 0 = Anomaly
# Prediction: 0 = Normal, 1 = Anomaly

actual_labels = (y_test == 0).astype(int)

accuracy = accuracy_score(actual_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

In [ ]:
# 13. Confusion Matrix and Classification Report
print("Confusion Matrix:")
print(confusion_matrix(actual_labels, predicted_labels))

print(classification_report(
    actual_labels,
    predicted_labels,
    target_names=['Normal', 'Anomaly']
))

In [ ]:
# 14. Plot Reconstruction Errors
plt.hist(reconstruction_errors.numpy(), bins=50,
         label='Normal Training Errors')

plt.axvline(threshold, linestyle='--', label='Threshold')
plt.title("Reconstruction Error Distribution")
plt.xlabel("Reconstruction Error")
plt.ylabel("Samples")
plt.legend()
plt.show()

In [ ]:
# 15. Final Results
print("ECG ANOMALY DETECTION - FINAL RESULTS")
print("Training Samples:", len(x_train_scaled))
print("Testing Samples:", len(x_test_scaled))
print("Anomaly Threshold:", round(threshold, 6))
print("Accuracy:", round(accuracy * 100, 2), "%")

print("Predicted Normal ECGs:", np.sum(predicted_labels == 0))
print("Predicted Anomalous ECGs:", np.sum(predicted_labels == 1))